In [1]:
import sys
import importlib
import warnings
import random
from collections import defaultdict
import pickle
import os
import re

import torch
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


sys.path.append("A:/Masters/Thesis/Dr. Mohsen Rashwan/projects/")

from stt_benchmarking.utils import helpers, text_processing, metrics as mt, validate
from stt_benchmarking.etl import load_data
from stt_benchmarking.models.speech_to_text.arabic import ensemble, rover
from stt_benchmarking.models.llms import reinforcer



warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option("display.max_colwidth", None)
random.seed(42)

print(device)

e:\Masters\Thesis\Dr. Mohsen Rashwan\projects\.venv\lib\site-packages\torch\backends\__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  self.setter(val)


cuda


In [ ]:
test_cv_path = "../../data/mozilla_commonvoice/cv-corpus/ar/test.tsv"
audio_paths = "../../data/mozilla_commonvoice/cv-corpus/ar/clips"
test_df = pd.read_csv(test_cv_path, sep="\t")

In [2]:
cv_audios = load_data.load_mozilla_cv(df=test_df, audio_path=audio_paths)

NameError: name 'test_df' is not defined

In [2]:
def save_bootstrap_samples(samples, filepath="../../reports/phase_1_4/lists_info/mozilla_cv/bootstrap_samples.pkl"):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'wb') as f:
        pickle.dump(samples, f)

def load_bootstrap_samples(filepath="../../reports/phase_1_4/lists_info/mozilla_cv/bootstrap_samples.pkl"):
    with open(filepath, 'rb') as f:
        return pickle.load(f)

In [5]:
save_bootstrap_samples(cv_audios, filepath="../../reports/phase_1_4/lists_info/mozilla_cv/cv_audios.pkl")

In [3]:
cv_audios = load_bootstrap_samples("../../results/mozilla_common_voice/audios/cv_audios.pkl")

# Models

In [4]:
import json
import inspect
import os

def save_model_info(obj, attr_name="samples_info", save_dir="../../reports/phase_1_4/models_info/mozilla_cv", is_llm=False, level=0):
    """
    Save the model's info dictionary to a JSON file.
    
    Parameters
    ----------
    obj : object
        Class object containing the info dictionary (e.g., seamless_m4t).
    attr_name : str, optional
        The attribute name holding the dictionary (default: "samples_info").
    save_dir : str, optional
        Path to directory where file will be saved.
    """
    # Ensure save directory exists
    os.makedirs(save_dir, exist_ok=True)

    # Get class object name
    obj_name = [name for name, val in inspect.currentframe().f_back.f_locals.items() if val is obj]
    obj_name = obj_name[0] if obj_name else obj.__class__.__name__

    # Extract info
    if not hasattr(obj, attr_name):
        raise AttributeError(f"Object has no attribute '{attr_name}'")
    
    info_dict = getattr(obj, attr_name)

    # Save as JSON
    if is_llm:
        save_path = os.path.join(save_dir, f"{obj_name}_llm_{level}.json")
    else:
        save_path = os.path.join(save_dir, f"{obj_name}_{level}.json")
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(info_dict, f, ensure_ascii=False, indent=2)

def save_list_info(list_obj, filename, save_dir="../../reports/phase_1_4/lists_info/mozilla_cv", is_llm=False):
    """
    Save a list of dictionaries to a JSON file.

    Parameters
    ----------
    list_obj : list[dict]
        The list of dictionaries to save.
    filename : str
        Base name for the saved file (without extension).
    save_dir : str, optional
        Directory where the file will be saved.
    """
    os.makedirs(save_dir, exist_ok=True)

    if is_llm:
        save_path = os.path.join(save_dir, f"{filename}_llm_.json")
    else:
        save_path = os.path.join(save_dir, f"{filename}.json")

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(list_obj, f, ensure_ascii=False, indent=2)

## Cohere

In [ ]:
from stt_benchmarking.models.speech_to_text.arabic import cohere_arabic
importlib.reload(cohere_arabic)

cohere = cohere_arabic.CohereArabicInference(device=device)
cohere.run_inference_one_by_one(cv_audios, substitute=False, normalize_final_letters=True)
save_list_info(cohere.samples_info, "cohere_samples")
cohere.overall_metrics

Loading weights: 100%|██████████| 2150/2150 [00:05<00:00, 403.88it/s]
2026-07-13 14:48:32 - [stt_benchmarking.models.speech_to_text.arabic._load_model:44] INFO - Loaded Cohere Transcribe Arabic (07-2026)
2026-07-13 14:48:32 - [stt_benchmarking.models.speech_to_text.arabic.run_inference_one_by_one:49] INFO - Using substitute: False, Replace Final Char: True
Processing Records: 100%|██████████| 10500/10500 [33:30<00:00,  5.22it/s] 
2026-07-13 15:22:03 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 33 minutes 30 seconds


{'word_error_rate': {'wer (%)': 5.62,
  'word_similarity_ratio (%)': 97.19,
  'word_edit_distance': 2995,
  'substitutions': 2504,
  'deletions': 294,
  'insertions': 197,
  'hits': 50466,
  'ref_length': 53264,
  'hyp_length': 53167},
 'character_error_rate': {'cer (%)': 1.57,
  'char_similarity_ratio (%)': 99.21,
  'char_edit_distance': 4128,
  'substitutions': 1411,
  'deletions': 1552,
  'insertions': 1165,
  'hits': 260056,
  'ref_length': 263019,
  'hyp_length': 262632}}

## HUBERT

In [6]:
from stt_benchmarking.models.speech_to_text.arabic import hubert
importlib.reload(hubert)

hubert_object = hubert.HubertInference(device=device)
hubert_object.run_inference_one_by_one(cv_audios, substitute=False, normalize_final_letters=True)
save_list_info(hubert_object.samples_info, "hubert_samples")
hubert_object.overall_metrics

Running on Windows; torchaudio backend switching is not available in this version.
Running on Windows; torchaudio backend switching is not available in this version.
During parameter transfer to HubertModel(
  (feature_extractor): HubertFeatureEncoder(
    (conv_layers): ModuleList(
      (0): HubertLayerNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
      (1-4): 4 x HubertLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
      (5-6): 2 x HubertLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection):

{'word_error_rate': {'wer (%)': 7.87,
  'word_similarity_ratio (%)': 96.07,
  'word_edit_distance': 4194,
  'substitutions': 3413,
  'deletions': 312,
  'insertions': 469,
  'hits': 49539,
  'ref_length': 53264,
  'hyp_length': 53421},
 'character_error_rate': {'cer (%)': 2.2,
  'char_similarity_ratio (%)': 98.9,
  'char_edit_distance': 5781,
  'substitutions': 1853,
  'deletions': 2164,
  'insertions': 1764,
  'hits': 259002,
  'ref_length': 263019,
  'hyp_length': 262619}}

## SpeechBrain

In [8]:
from stt_benchmarking.models.speech_to_text.arabic import speechbrain
importlib.reload(speechbrain)

wav2vec_sb_object = speechbrain.SpeechBrainInference(device=device)
wav2vec_sb_object.run_inference_one_by_one(cv_audios, substitute=False, normalize_final_letters=True)
save_list_info(wav2vec_sb_object.samples_info, "wav2vec_sb_samples")
wav2vec_sb_object.overall_metrics

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

speechbrain.lobes.models.huggingface_transformers.huggingface - Wav2Vec2Model is frozen.


model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

2026-07-13 14:03:52 - [stt_benchmarking.models.speech_to_text._load_model:43] INFO - Loaded model speechbrain/asr-wav2vec2-commonvoice-14-ar
2026-07-13 14:03:52 - [stt_benchmarking.models.speech_to_text.run_inference_one_by_one:55] INFO - Using substitute: False, Replace Final Char: True
Processing Records: 100%|██████████| 10500/10500 [04:34<00:00, 38.30it/s]
2026-07-13 14:08:28 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 4 minutes 35 seconds


{'word_error_rate': {'wer (%)': 29.17,
  'word_similarity_ratio (%)': 85.34,
  'word_edit_distance': 15537,
  'substitutions': 13154,
  'deletions': 1475,
  'insertions': 908,
  'hits': 38635,
  'ref_length': 53264,
  'hyp_length': 52697},
 'character_error_rate': {'cer (%)': 9.72,
  'char_similarity_ratio (%)': 95.12,
  'char_edit_distance': 25565,
  'substitutions': 8095,
  'deletions': 9726,
  'insertions': 7744,
  'hits': 245198,
  'ref_length': 263019,
  'hyp_length': 261037}}

## XLSR

In [10]:
from stt_benchmarking.models.speech_to_text.arabic import xlsr_53
importlib.reload(xlsr_53)

xlsr_object = xlsr_53.XLSRInference(device=device)
xlsr_object.run_inference_one_by_one(cv_audios, substitute=False, normalize_final_letters=True)
save_list_info(xlsr_object.samples_info, "xlsr_samples")
xlsr_object.overall_metrics

`torch_dtype` is deprecated! Use `dtype` instead!
2026-07-13 14:09:52 - [stt_benchmarking.models.speech_to_text._load_model:66] INFO - Loaded Wav2Vec2 XLSR-53 XLSR-53 Model
2026-07-13 14:09:52 - [stt_benchmarking.models.speech_to_text.run_inference_one_by_one:78] INFO - Using substitute: False, Replace Final Char: True
Processing Records: 100%|██████████| 10500/10500 [05:19<00:00, 32.90it/s]
2026-07-13 14:15:13 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 5 minutes 20 seconds


{'word_error_rate': {'wer (%)': 22.87,
  'word_similarity_ratio (%)': 88.53,
  'word_edit_distance': 12183,
  'substitutions': 10621,
  'deletions': 937,
  'insertions': 625,
  'hits': 41706,
  'ref_length': 53264,
  'hyp_length': 52952},
 'character_error_rate': {'cer (%)': 6.49,
  'char_similarity_ratio (%)': 96.75,
  'char_edit_distance': 17069,
  'substitutions': 7361,
  'deletions': 5536,
  'insertions': 4172,
  'hits': 250122,
  'ref_length': 263019,
  'hyp_length': 261655}}

## W2v-BERT-AR

In [11]:
from stt_benchmarking.models.speech_to_text.arabic import w2v_bert_ar
importlib.reload(w2v_bert_ar)

w2v_bert_object = w2v_bert_ar.w2vBERTInference(device=device)
w2v_bert_object.run_inference_one_by_one(cv_audios, substitute=False, normalize_final_letters=True)
save_list_info(w2v_bert_object.samples_info, "w2vbert_samples")
w2v_bert_object.overall_metrics

`torch_dtype` is deprecated! Use `dtype` instead!
2026-07-13 14:16:31 - [stt_benchmarking.models.speech_to_text._load_model:48] INFO - Loaded model w2v Bert Arabic
2026-07-13 14:16:31 - [stt_benchmarking.models.speech_to_text.run_inference_one_by_one:60] INFO - Using substitute: False, Replace Final Char: True
Processing Records: 100%|██████████| 10500/10500 [06:27<00:00, 27.12it/s]
2026-07-13 14:22:59 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 6 minutes 28 seconds


{'word_error_rate': {'wer (%)': 24.03,
  'word_similarity_ratio (%)': 87.98,
  'word_edit_distance': 12802,
  'substitutions': 10979,
  'deletions': 944,
  'insertions': 879,
  'hits': 41341,
  'ref_length': 53264,
  'hyp_length': 53199},
 'character_error_rate': {'cer (%)': 6.65,
  'char_similarity_ratio (%)': 96.68,
  'char_edit_distance': 17483,
  'substitutions': 5913,
  'deletions': 5734,
  'insertions': 5836,
  'hits': 251372,
  'ref_length': 263019,
  'hyp_length': 263121}}

## Whisper

In [6]:
from stt_benchmarking.models.speech_to_text.arabic import whisper
importlib.reload(whisper)

whisper_v2 = whisper.FasterWhisperInference(device=device, model_version="v2", batch_size=4)
whisper_v2.run_inference_one_by_one(cv_audios, substitute=False, normalize_final_letters=True)
save_list_info(whisper_v2.samples_info, "whisper_v2_samples")
whisper_v2.overall_metrics

2026-07-15 12:40:54 - [stt_benchmarking.models.speech_to_text._load_model:53] INFO - Loaded FasterWhisper V2 Model
2026-07-15 12:40:54 - [stt_benchmarking.models.speech_to_text._load_pipeline:64] INFO - Loaded Whisper V2 Pipeline
2026-07-15 12:40:54 - [stt_benchmarking.models.speech_to_text.run_inference_one_by_one:77] INFO - Using substitute: False, Replace Final Char: True
Processing Records: 100%|██████████| 10500/10500 [2:00:20<00:00,  1.45it/s] 
2026-07-15 14:41:16 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 120 minutes 21 seconds


{'word_error_rate': {'wer (%)': 18.47,
  'word_similarity_ratio (%)': 90.74,
  'word_edit_distance': 9840,
  'substitutions': 8395,
  'deletions': 827,
  'insertions': 618,
  'hits': 44042,
  'ref_length': 53264,
  'hyp_length': 53055},
 'character_error_rate': {'cer (%)': 6.06,
  'char_similarity_ratio (%)': 96.97,
  'char_edit_distance': 15948,
  'substitutions': 6618,
  'deletions': 4119,
  'insertions': 5211,
  'hits': 252282,
  'ref_length': 263019,
  'hyp_length': 264111}}

In [5]:
from stt_benchmarking.models.speech_to_text.arabic import whisper
importlib.reload(whisper)

whisper_v3 = whisper.FasterWhisperInference(device=device, model_version="v3", batch_size=8)
whisper_v3.run_inference_one_by_one(cv_audios, substitute=False, normalize_final_letters=True)
save_list_info(whisper_v3.samples_info, "whisper_v3_samples")
whisper_v3.overall_metrics

2026-07-13 16:40:58 - [stt_benchmarking.models.speech_to_text._load_model:53] INFO - Loaded FasterWhisper V3 Model
2026-07-13 16:40:58 - [stt_benchmarking.models.speech_to_text._load_pipeline:64] INFO - Loaded Whisper V3 Pipeline
2026-07-13 16:40:58 - [stt_benchmarking.models.speech_to_text.run_inference_one_by_one:77] INFO - Using substitute: False, Replace Final Char: True
Processing Records: 100%|██████████| 10500/10500 [1:55:24<00:00,  1.52it/s] 
2026-07-13 18:36:24 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 115 minutes 26 seconds


{'word_error_rate': {'wer (%)': 14.77,
  'word_similarity_ratio (%)': 92.6,
  'word_edit_distance': 7867,
  'substitutions': 6741,
  'deletions': 685,
  'insertions': 441,
  'hits': 45838,
  'ref_length': 53264,
  'hyp_length': 53020},
 'character_error_rate': {'cer (%)': 4.69,
  'char_similarity_ratio (%)': 97.65,
  'char_edit_distance': 12340,
  'substitutions': 4984,
  'deletions': 3700,
  'insertions': 3656,
  'hits': 254335,
  'ref_length': 263019,
  'hyp_length': 262975}}

## Fastconformer

In [12]:
from stt_benchmarking.models.speech_to_text.arabic import fastconformer_nvidia
importlib.reload(fastconformer_nvidia)

nvidia_object = fastconformer_nvidia.FastConformerInference(device=device)
nvidia_object.run_inference_one_by_one(cv_audios, substitute=False, normalize_final_letters=True)
save_list_info(nvidia_object.samples_info, "fastconformer_samples")
nvidia_object.overall_metrics

[NeMo W 2026-07-13 14:23:05 model_utils:535] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': True, 'num_workers': 8, 'pin_memory': True, 'max_duration': 20, 'min_duration': 0.5, 'is_tarred': True, 'tarred_audio_filepaths': '???', 'shuffle_n': 2048, 'bucketing_strategy': 'fully_randomized', 'bucketing_batch_size': None}
     Reason: Missing mandatory value: train_ds.manifest_filepath
        full_key: train_ds.manifest_filepath
        object_type=dict.
[NeMo W 2026-07-13 14:23:05 model_utils:535] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': False, 'use_start_end_token': False, 'num_workers': 8, 'pin_memory': True}
     Reason: Missing mandatory value: validation_ds.manifest_filepath
        full_key: validation_ds.manifest_filepath
        object_type=dict.
[NeMo W 2026-07-13 14:23:05 model_utils:535] Skipped conversion for config/

[NeMo I 2026-07-13 14:23:06 mixins:181] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-07-13 14:23:06 model_utils:535] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': True, 'num_workers': 8, 'pin_memory': True, 'max_duration': 20, 'min_duration': 0.5, 'is_tarred': True, 'tarred_audio_filepaths': '???', 'shuffle_n': 2048, 'bucketing_strategy': 'fully_randomized', 'bucketing_batch_size': None}
     Reason: Missing mandatory value: train_ds.manifest_filepath
        full_key: train_ds.manifest_filepath
        object_type=dict.
[NeMo W 2026-07-13 14:23:06 model_utils:535] Skipped conversion for config/subconfig:
    {'manifest_filepath': '???', 'sample_rate': 16000, 'batch_size': 16, 'shuffle': False, 'use_start_end_token': False, 'num_workers': 8, 'pin_memory': True}
     Reason: Missing mandatory value: validation_ds.manifest_filepath
        full_key: validation_ds.manifest_filepath
        object_type=dict.
[NeMo W 2026-07-13 14:23:06 model_utils:535] Skipped conversion for config/

[NeMo I 2026-07-13 14:23:07 features:305] PADDING: 0
[NeMo I 2026-07-13 14:23:07 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-07-13 14:23:07 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-07-13 14:23:08 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-07-13 14:23:08 save_restore_connector:275] Model EncDecHybridRNNTCTCBPEModel was successfully restored from C:\Users\school.RABAH-DESKTOP\.cache\huggingface\hub\models--nvidia--stt_ar_fastconformer_hybrid_large_pcd_v1.0\snapshots\7f32349d952f42a28dce979ba73270aa2bbdfa89\stt_ar_fastconformer_hybrid_large_pcd_v1.0.nemo.


2026-07-13 14:23:08 - [stt_benchmarking.models.speech_to_text._load_model:41] INFO - Loaded model stt conformer hybrid
2026-07-13 14:23:08 - [stt_benchmarking.models.speech_to_text.run_inference_one_by_one:53] INFO - Using substitute: False, Replace Final Char: True
Processing Records: 100%|██████████| 10500/10500 [11:58<00:00, 14.61it/s]
2026-07-13 14:35:08 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 12 minutes 0 seconds


{'word_error_rate': {'wer (%)': 7.25,
  'word_similarity_ratio (%)': 96.37,
  'word_edit_distance': 3863,
  'substitutions': 3222,
  'deletions': 421,
  'insertions': 220,
  'hits': 49621,
  'ref_length': 53264,
  'hyp_length': 53063},
 'character_error_rate': {'cer (%)': 2.22,
  'char_similarity_ratio (%)': 98.88,
  'char_edit_distance': 5852,
  'substitutions': 1998,
  'deletions': 2578,
  'insertions': 1276,
  'hits': 258443,
  'ref_length': 263019,
  'hyp_length': 261717}}

## Seamless M4t v2

In [5]:
from stt_benchmarking.models.speech_to_text.arabic import seamlessm4t_v2
importlib.reload(seamlessm4t_v2)

seamless_m4t = seamlessm4t_v2.SeamlessM4TFullInterface(device=device)
seamless_m4t.run_inference_optimized(cv_audios, substitute=False, normalize_final_letters=True)
save_list_info(seamless_m4t.samples_info, "seamless_samples")
seamless_m4t.overall_metrics

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-07-13 15:30:06 - [stt_benchmarking.models.speech_to_text.arabic._load_model:86] INFO - Loaded Seamless M4T V2 Model
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
2026-07-13 15:30:13 - [stt_benchmarking.models.speech_to_text.arabic._load_pipeline:112] INFO - Loaded Seamless M4T V2 Pipeline (20s chunks / 1s stride)
2026-07-13 15:30:13 - [stt_benchmarking.models.speech_to_text.arabic.run_inference_optimized:282] INFO - Using substitute: False, Replace Final Char: True
2026-07-13 15:30:13 - [stt_benchmarking.models.speech_to_text.arabic.run_inference_optimized:290] INFO - Running one-by-one inference on 10500 short files (<30.0s)
Processing Records: 100%|██████████| 10500/10500 [1:05:17<00:00,  2.68it/s]
2026-07-13 16:35:31 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_one_by_one: 65 minutes 17 seconds
2026-07-13 16:35:31 - [stt_benchmarking.utils.decorators.execution_time_wrapper:22] INFO - Execution time for function run_inference_optimized: 65 minutes 18 seconds


{'word_error_rate': {'wer (%)': 10.84,
  'word_similarity_ratio (%)': 94.59,
  'word_edit_distance': 5774,
  'substitutions': 4983,
  'deletions': 334,
  'insertions': 457,
  'hits': 47947,
  'ref_length': 53264,
  'hyp_length': 53387},
 'character_error_rate': {'cer (%)': 3.55,
  'char_similarity_ratio (%)': 98.23,
  'char_edit_distance': 9340,
  'substitutions': 3507,
  'deletions': 2327,
  'insertions': 3506,
  'hits': 257185,
  'ref_length': 263019,
  'hyp_length': 264198}}

In [4]:
import os
import json

def load_samples_info(file_name, file_dr="../../results/mozilla_common_voice/audios"):
    """
    Load a JSON file from the given directory and normalize paths to start with ../../data
    
    Args:
        file_name (str): Name of the JSON file (with .json extension).
        file_dr (str): Directory where the file is stored.

    Returns:
        dict: Parsed JSON data with normalized paths as keys.
    """
    file_path = os.path.join(file_dr, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    normalized_data = {}
    for key, value in data.items():        
        normalized_data[key] = value
    
    return normalized_data

def load_list_info(filename, load_dir="../../reports/phase_1_4/lists_info/ultimate_last"):
    """
    Load a list of dictionaries saved as JSON (via save_list_info).
    
    Parameters
    ----------
    filename : str
        The name of the JSON file (e.g., 'ensemble_samples_0.json').
    load_dir : str, optional
        Path to the directory containing the saved file.
        
    Returns
    -------
    list
        Loaded list of dictionaries.
    """
    load_path = os.path.join(load_dir, filename)
    if not os.path.exists(load_path):
        raise FileNotFoundError(f"File not found: {load_path}")
    
    with open(load_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    if not isinstance(data, list):
        raise TypeError(f"Expected list, got {type(data).__name__} from {filename}")
    
    return data

In [5]:
def collect_model_metrics(models_with_names):
    """
    Collect metrics from multiple model samples_info objects into a unified DataFrame.
    
    Parameters
    ----------
    models_with_names : list[tuple[str, dict]]
        List of tuples (model_name, samples_info_dict).
    
    Returns
    -------
    pd.DataFrame
        DataFrame containing all model metrics and metadata.
    """
    records = []
    for model_name, samples_info in models_with_names:
        for audio_id, content in samples_info.items():
            metrics = content["metrics"]
            wer_data = metrics["word_error_rate"]
            cer_data = metrics["character_error_rate"]

            wer = wer_data["wer (%)"]
            cer = cer_data["cer (%)"]

            if model_name in {"ensemble", "hybrid_ensemble", "rover"}:
                inference_time = content.get("fusion_time")
            else:
                inference_time = content.get("inference_time")

            records.append({
                "audio_id": audio_id,
                "model": model_name,
                "raw_transcription": content.get("raw_transcription"),
                "normalized_transcription": content.get("normalized_transcription"),
                "raw_prediction": content.get("raw_prediction"),
                "normalized_prediction": content.get("normalized_prediction"),
                "inference_time": inference_time,
                "rtf": content.get("rtf"),
                "fusion_time": content.get("fusion_time"),
                "duration": content.get("duration"),
                "wer": wer,
                "substitutions_w": wer_data["substitutions"],
                "deletions_w": wer_data["deletions"],
                "insertions_w": wer_data["insertions"],
                "hits_w": wer_data["hits"],
                "cer": cer,
                "substitutions_c": cer_data["substitutions"],
                "deletions_c": cer_data["deletions"],
                "insertions_c": cer_data["insertions"],
                "hits_c": cer_data["hits"],
            })

    return pd.DataFrame(records)

def filter_audio_records(df_samples, ensemble_name="ensemble"):
    empty = df_samples["normalized_prediction"].fillna("").str.strip() == ""
    is_ens = df_samples["model"] == ensemble_name
    ens_empty = (empty & is_ens).groupby(df_samples["audio_id"]).transform("any")

    empty_gt = df_samples["normalized_transcription"].fillna("").str.strip() == ""
    gt_empty = empty_gt.groupby(df_samples["audio_id"]).transform("any")

    drop = ens_empty | gt_empty
    df_filtered = df_samples[~drop].reset_index(drop=True)

    print(f"Audios dropped (empty ground truth): {df_samples.loc[gt_empty, 'audio_id'].nunique()}")
    print(f"Audios dropped (empty {ensemble_name}): {df_samples.loc[ens_empty, 'audio_id'].nunique()}")
    print(f"Audios: {df_samples['audio_id'].nunique()} -> {df_filtered['audio_id'].nunique()}")
    return df_filtered

def evaluate_absolute_metrics(df_filtered):
    """
    Compute absolute WER/CER metrics for each model against human references.
    """
    agg_results = []
    for model, g in df_filtered.groupby("model"):
        refs = g["normalized_transcription"].tolist()
        hyps = g["normalized_prediction"].tolist()

        duration_mean = pd.Series(g.get("duration", [])).dropna().mean()
        inference_time_mean = pd.Series(g.get("inference_time", [])).dropna().mean()
        rtf_mean = pd.Series(g.get("rtf", [])).dropna().mean()
        fusion_time_mean = pd.Series(g.get("fusion_time", [])).dropna().mean()
        eval_result = mt.BasicSTTMetrics.evaluate(refs=refs, hyps=hyps)
        word_metrics = eval_result['word_error_rate']
        char_metrics = eval_result['character_error_rate']

        agg_results.append({
            "model": model,
            "WER (%)": word_metrics["wer (%)"],
            "w_substitutions": word_metrics["substitutions"],
            "w_deletions": word_metrics["deletions"],
            "w_insertions": word_metrics["insertions"],
            "w_hits": word_metrics["hits"],
            "word_edit_distance": word_metrics["word_edit_distance"],
            "word_similarity_ratio (%)": word_metrics["word_similarity_ratio (%)"],
            "w_ref_length": word_metrics["ref_length"],
            "w_hyp_length": word_metrics["hyp_length"],

            "CER (%)": char_metrics["cer (%)"],
            "c_substitutions": char_metrics["substitutions"],
            "c_deletions": char_metrics["deletions"],
            "c_insertions": char_metrics["insertions"],
            "c_hits": char_metrics["hits"],
            "char_edit_distance": char_metrics["char_edit_distance"],
            "char_similarity_ratio (%)": char_metrics["char_similarity_ratio (%)"],
            "c_ref_length": char_metrics["ref_length"],
            "c_hyp_length": char_metrics["hyp_length"],

            "duration_mean": duration_mean if not pd.isna(duration_mean) else None,
            "inference_time_mean": inference_time_mean if not pd.isna(inference_time_mean) else None,
            "rtf_mean": rtf_mean if not pd.isna(rtf_mean) else None,
            "inference_time_total_h": (inference_time_mean * len(g) / 3600) if not pd.isna(inference_time_mean) else None,
            "fusion_time_mean": fusion_time_mean
            
        })

    df_eval = pd.DataFrame(agg_results)
    df_eval = df_eval.sort_values(
        by=["WER (%)", "word_similarity_ratio (%)"],
        ascending=[True, False]
    ).reset_index(drop=True)
    return df_eval

def evaluate_relative_metrics(df_filtered, model_name, exclude_models=[]):
    """
    Compute relative WER/CER metrics for each model using ensemble outputs as reference.
    """
    agg_results = []

    ensemble_df = (
        df_filtered[df_filtered['model'] == model_name]
        [["audio_id", "normalized_prediction"]]
        .reset_index(drop=True)
    )
    ensemble_df['_order'] = range(len(ensemble_df))

    assert ensemble_df["audio_id"].is_unique, "Duplicate audio_id in ensemble."
    assert ensemble_df["normalized_prediction"].notna().all(), "Ensemble contains None normalized_prediction."

    models_to_compare = df_filtered[~df_filtered['model'].isin(exclude_models)]

    for model, g in models_to_compare.groupby("model"):
        g = g[["audio_id", "normalized_prediction"]].reset_index(drop=True)

        assert g["audio_id"].is_unique, f"Duplicate audio_id in model {model}."
        assert g["normalized_prediction"].notna().all(), f"Model {model} contains None normalized_prediction."

        merged = ensemble_df.merge(g, on="audio_id", suffixes=("_ref", "_hyp"), how="inner")
        merged = merged.sort_values("_order").reset_index(drop=True)
        assert len(merged) == len(ensemble_df), f"Alignment mismatch for {model}: {len(merged)} vs {len(ensemble_df)}."

        refs = merged["normalized_prediction_ref"].tolist()
        hyps = merged["normalized_prediction_hyp"].tolist()
        assert len(refs) > 0, f"No valid reference for model {model}."

        eval_result = mt.BasicSTTMetrics.evaluate(refs=refs, hyps=hyps)
        word_metrics = eval_result['word_error_rate']
        char_metrics = eval_result['character_error_rate']

        agg_results.append({
            "model": model,
            "WER (%)": word_metrics["wer (%)"],
            "w_substitutions": word_metrics["substitutions"],
            "w_deletions": word_metrics["deletions"],
            "w_insertions": word_metrics["insertions"],
            "w_hits": word_metrics["hits"],
            "word_edit_distance": word_metrics["word_edit_distance"],
            "word_similarity_ratio (%)": word_metrics["word_similarity_ratio (%)"],
            "w_ref_length": word_metrics["ref_length"],
            "w_hyp_length": word_metrics["hyp_length"],

            "CER (%)": char_metrics["cer (%)"],
            "c_substitutions": char_metrics["substitutions"],
            "c_deletions": char_metrics["deletions"],
            "c_insertions": char_metrics["insertions"],
            "c_hits": char_metrics["hits"],
            "char_edit_distance": char_metrics["char_edit_distance"],
            "char_similarity_ratio (%)": char_metrics["char_similarity_ratio (%)"],
            "c_ref_length": char_metrics["ref_length"],
            "c_hyp_length": char_metrics["hyp_length"],
        })

    df_relative = pd.DataFrame(agg_results)
    df_relative = df_relative.sort_values(
        by=["WER (%)", "word_similarity_ratio (%)"],
        ascending=[True, False]
    ).reset_index(drop=True)
    return df_relative

In [6]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau, pearsonr

# assumes `mt` (your metrics module) is importable in the calling scope,
# i.e. mt.BasicSTTMetrics.evaluate(refs=..., hyps=...) as used elsewhere.


def add_pseudo_reference_wer(df_filtered, ensemble, transcript_key="normalized_prediction"):
    """
    Add per-sample pseudo-reference WER/CER.

    ensemble : dict[audio_id -> dict] with the fused transcript under `transcript_key`.
    """
    ens = {aid: v[transcript_key] for aid, v in ensemble.items()}

    refs = df_filtered["audio_id"].map(ens)
    hyps = df_filtered[transcript_key]

    pseudo_wer, pseudo_cer = [], []
    for ref, hyp in zip(refs, hyps):
        if not isinstance(ref, str) or not ref.strip() or not isinstance(hyp, str):
            pseudo_wer.append(np.nan)
            pseudo_cer.append(np.nan)
            continue
        r = mt.BasicSTTMetrics.evaluate(refs=ref, hyps=hyp)
        pseudo_wer.append(r["word_error_rate"]["wer (%)"])
        pseudo_cer.append(r["character_error_rate"]["cer (%)"])

    df = df_filtered.copy()
    df["pseudo_wer"] = pseudo_wer
    df["pseudo_cer"] = pseudo_cer

    missing = refs.isna().sum()
    if missing:
        print(f"Rows with no ensemble reference: {missing}")
    return df

def _corr(x, y):
    if len(x) < 3 or pd.Series(x).nunique() < 2 or pd.Series(y).nunique() < 2:
        return None
    sp = spearmanr(x, y)
    kt = kendalltau(x, y)
    pr = pearsonr(x, y)
    return {
        "pearson": pr.statistic, "pearson_p": pr.pvalue,
        "spearman": sp.statistic, "spearman_p": sp.pvalue,
        "kendall": kt.statistic, "kendall_p": kt.pvalue,
    }


def per_sample_correlation(df_pseudo, exclude_models=("ensemble", "rover"),
                           metric_pair=("wer", "pseudo_wer"), min_pairs=30):
    """
    Option A. Per-sample WER tracking: correlate true WER vs pseudo-reference WER.

    Returns a df with one row per model (within-model, across samples) plus an
    'ALL' row (pooled over every sample-model pair, the direct NoRefER analogue).
    """
    true_col, pseudo_col = metric_pair
    sub = df_pseudo[~df_pseudo["model"].isin(exclude_models)].copy()
    rows = []

    for model, g in sub.groupby("model"):
        d = g[[true_col, pseudo_col]].dropna()
        if len(d) < min_pairs:
            continue
        c = _corr(d[true_col].values, d[pseudo_col].values)
        if c:
            rows.append({"model": model, "n": len(d), **c})

    d_all = sub[[true_col, pseudo_col]].dropna()
    c_all = _corr(d_all[true_col].values, d_all[pseudo_col].values)
    if c_all:
        rows.append({"model": "ALL (pooled)", "n": len(d_all), **c_all})

    return pd.DataFrame(rows).sort_values("spearman", ascending=False).reset_index(drop=True)


def per_sample_ranking(df_pseudo, exclude_models=("ensemble", "rover"),
                       metric_cols=("wer", "pseudo_wer"),
                       tiebreak_cols=("cer", "pseudo_cer"),
                       min_models=3):
    """
    Per-sample ranking agreement, WER primary with CER as tiebreak only.

    For each sample, systems are ordered by true (WER, then CER) and by
    pseudo (WER, then CER); CER is consulted solely to break exact WER ties,
    so the axis remains a WER ranking, not a WER+CER blend. The two orderings
    are correlated within the sample, then aggregated across samples.

    Returns a one-row summary df with mean/median/std of Spearman and Kendall,
    usable/dropped sample counts, and the residual WER tie fraction.
    """
    true_w, pseudo_w = metric_cols
    true_c, pseudo_c = tiebreak_cols
    sub = df_pseudo[~df_pseudo["model"].isin(exclude_models)].copy()

    sp_vals, kt_vals, tie_fracs = [], [], []
    n_dropped = 0

    for audio_id, g in sub.groupby("audio_id"):
        d = g[[true_w, pseudo_w, true_c, pseudo_c]].dropna()
        if len(d) < min_models:
            n_dropped += 1
            continue

        # residual WER-only tie fraction, before CER breaks them (diagnostic)
        _, counts = np.unique(d[true_w].values, return_counts=True)
        tie_fracs.append(1.0 - len(counts) / len(d))

        # composite ranks: WER primary, CER tiebreak. lexsort keys are
        # applied last-key-first, so primary key goes last.
        true_rank = np.lexsort((d[true_c].values, d[true_w].values)).argsort()
        pseudo_rank = np.lexsort((d[pseudo_c].values, d[pseudo_w].values)).argsort()

        if pd.Series(true_rank).nunique() < 2 or pd.Series(pseudo_rank).nunique() < 2:
            n_dropped += 1
            continue

        sp_vals.append(spearmanr(true_rank, pseudo_rank).statistic)
        kt_vals.append(kendalltau(true_rank, pseudo_rank).statistic)

    summary = {
        "n_samples_used": len(sp_vals),
        "n_samples_dropped": n_dropped,
        "mean_wer_ties": float(np.mean(tie_fracs)) if tie_fracs else np.nan,
        "spearman_mean": float(np.nanmean(sp_vals)) if sp_vals else np.nan,
        "spearman_median": float(np.nanmedian(sp_vals)) if sp_vals else np.nan,
        "spearman_std": float(np.nanstd(sp_vals)) if sp_vals else np.nan,
        "kendall_mean": float(np.nanmean(kt_vals)) if kt_vals else np.nan,
        "kendall_median": float(np.nanmedian(kt_vals)) if kt_vals else np.nan,
        "kendall_std": float(np.nanstd(kt_vals)) if kt_vals else np.nan,
    }
    return pd.DataFrame([summary])


def ranking_correlation(df_absolute, df_relative, exclude_models=("ensemble", "rover")):
    """
    Option B (headline). Correlate the reference ranking against the
    pseudo-reference ranking over models, on full-set aggregate WER.

    df_absolute : output of evaluate_absolute_metrics
    df_relative : output of evaluate_relative_metrics
    Returns (merged_df, coefficients_dict).
    """
    a = df_absolute[["model", "WER (%)"]].rename(columns={"WER (%)": "wer_abs"})
    r = df_relative[["model", "WER (%)"]].rename(columns={"WER (%)": "wer_rel"})
    m = a.merge(r, on="model", how="inner")
    m = m[~m["model"].isin(exclude_models)].reset_index(drop=True)

    c = _corr(m["wer_abs"].values, m["wer_rel"].values)
    return m, c

In [7]:
seamless_samples = load_samples_info("seamless_samples.json")
whisper_v3_samples = load_samples_info("whisper_v3_samples.json")
whisper_v2_samples = load_samples_info("whisper_v2_samples.json")
nvidia_samples = load_samples_info("fastconformer_samples.json")
xlsr_samples = load_samples_info("xlsr_samples.json")
w2vbert_samples = load_samples_info("w2vbert_samples.json")
hubert_samples = load_samples_info("hubert_samples.json")
wav2vec_sb_samples = load_samples_info("wav2vec_sb_samples.json")
cohere_samples = load_samples_info("cohere_samples.json")

In [8]:
all_models = [
    seamless_samples,
    whisper_v3_samples,
    whisper_v2_samples,
    nvidia_samples,
    xlsr_samples,
    w2vbert_samples,
    hubert_samples,
    wav2vec_sb_samples,
    cohere_samples
]

model_names = [
    "seamless_m4t",
    "whisper_v3",
    "whisper_v2",
    "fastconformer",
    "xlsr",
    "w2vbert",
    "hubert",
    "wav2vec_sb",
    "cohere"
]

voters = ["seamless_m4t", "whisper_v3", "fastconformer", "w2vbert", "hubert", "wav2vec_sb", "cohere"]
voters_as_elements = [
    seamless_samples,
    whisper_v3_samples,
    nvidia_samples,
    w2vbert_samples,
    hubert_samples,
    wav2vec_sb_samples,
    cohere_samples]

# ROVER - Evaluation Protocol

In [ ]:
from stt_benchmarking.utils import evaluation_protocol
from stt_benchmarking.models.speech_to_text.arabic import rover
importlib.reload(evaluation_protocol)
importlib.reload(rover)

proto = evaluation_protocol.EvaluationProtocol(
    model_samples=dict(zip(model_names, all_models)),
    voters=voters,
    ensemble_factory=rover.ROVER,
    outdir="protocol_out/mozilla_commonvoice_final_rover",
    n_boot=10000, seed=42, min_pool_size=3,
    main_kwargs={"substitute": False, "normalize_final_letters": True},
    suffix="_rover_cv",
)
report = proto.run()

2026-09-10 12:56:38 - [stt_benchmarking.utils.evaluation_protocol.step0_decorrelation:265] INFO - Step 0 Finished.
2026-09-10 12:56:38 - [stt_benchmarking.models.speech_to_text.arabic.main:406] INFO - Using substitute: False, Replace Final Char: True
ROVER Fusion...: 100%|██████████| 10500/10500 [00:53<00:00, 196.46it/s]
2026-09-10 12:57:35 - [stt_benchmarking.models.speech_to_text.arabic.main:406] INFO - Using substitute: False, Replace Final Char: True
ROVER Fusion...: 100%|██████████| 10500/10500 [00:44<00:00, 234.87it/s]
2026-09-10 12:58:22 - [stt_benchmarking.models.speech_to_text.arabic.main:406] INFO - Using substitute: False, Replace Final Char: True
ROVER Fusion...: 100%|██████████| 10500/10500 [00:41<00:00, 250.31it/s]
2026-09-10 12:59:09 - [stt_benchmarking.models.speech_to_text.arabic.main:406] INFO - Using substitute: False, Replace Final Char: True
ROVER Fusion...: 100%|██████████| 10500/10500 [00:42<00:00, 248.53it/s]
2026-09-10 12:59:56 - [stt_benchmarking.models.speech


=== skipped_empty_gt ===
0

=== main_kwargs_applied ===
{
  "substitute": false,
  "normalize_final_letters": true
}

=== main_kwargs_dropped ===
[]

=== step0_cross_wer ===
               seamless_m4t  whisper_v3  whisper_v2  fastconformer   xlsr  w2vbert  hubert  wav2vec_sb  cohere
seamless_m4t           0.00       15.80       19.40          11.85  24.88    25.77   13.86       30.72   10.23
whisper_v3            15.80        0.00       14.77          15.00  25.50    26.23   17.06       31.71   13.88
whisper_v2            19.40       14.77        0.00          18.63  28.03    28.83   20.50       33.99   17.57
fastconformer         11.85       15.00       18.63           0.00  22.77    24.06   10.30       28.80    7.05
xlsr                  24.88       25.50       28.03          22.77   0.00    31.68   23.49       31.10   22.29
w2vbert               25.77       26.23       28.83          24.06  31.68     0.00   25.33       37.20   23.32
hubert                13.86       17.06       20

# Ensemble - Evaluation Protocol

In [11]:
from stt_benchmarking.utils import evaluation_protocol
from stt_benchmarking.models.speech_to_text.arabic import ensemble
importlib.reload(evaluation_protocol)
importlib.reload(ensemble)

proto = evaluation_protocol.EvaluationProtocol(
    model_samples=dict(zip(model_names, all_models)),
    voters=voters,
    ensemble_factory=ensemble.HybridEnsemble,
    outdir="protocol_out/cv_validated_ensemble",
    n_boot=10000, seed=42, min_pool_size=3,
    main_kwargs={"substitute": False, "normalize_final_letters": True},
    suffix="_validated_ensemble_cv",
)
report = proto.run()


2026-09-22 18:07:01 - [stt_benchmarking.utils.evaluation_protocol.step0_decorrelation:265] INFO - Step 0 Finished.
2026-09-22 18:07:01 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:26<00:00, 401.53it/s]
2026-09-22 18:07:28 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:21<00:00, 487.19it/s]
2026-09-22 18:07:51 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:21<00:00, 494.46it/s]
2026-09-22 18:08:14 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:21<00:00, 481.01it/s]
2026-09-22 18:08:37 - [stt_benchmarking.models.sp


=== skipped_empty_gt ===
0

=== main_kwargs_applied ===
{
  "substitute": false,
  "normalize_final_letters": true
}

=== main_kwargs_dropped ===
[]

=== step0_cross_wer ===
               seamless_m4t  whisper_v3  whisper_v2  fastconformer   xlsr  w2vbert  hubert  wav2vec_sb  cohere
seamless_m4t           0.00       15.80       19.40          11.85  24.88    25.77   13.86       30.72   10.23
whisper_v3            15.80        0.00       14.77          15.00  25.50    26.23   17.06       31.71   13.88
whisper_v2            19.40       14.77        0.00          18.63  28.03    28.83   20.50       33.99   17.57
fastconformer         11.85       15.00       18.63           0.00  22.77    24.06   10.30       28.80    7.05
xlsr                  24.88       25.50       28.03          22.77   0.00    31.68   23.49       31.10   22.29
w2vbert               25.77       26.23       28.83          24.06  31.68     0.00   25.33       37.20   23.32
hubert                13.86       17.06       20

In [9]:
from stt_benchmarking.utils import deployment_protocol as dp
from stt_benchmarking.models.speech_to_text.arabic import ensemble

importlib.reload(dp)

dp = dp.DeploymentProtocol.from_hypotheses(
    model_samples=dict(zip(model_names, all_models)),
    voters=model_names,
    ensemble_factory=ensemble.HybridEnsemble,
    main_kwargs={"substitute": False, "normalize_final_letters": True},
    n_boot=1000, seed=42, min_pool_size=3,
    outdir="deployment_out/cv_validated_all_voting_ensemble", suffix="cv_validated_all_voting_ensemble")
    
r = dp.run(weighting="filtration")

2026-09-22 19:01:38 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function cross_wer: 0 minutes 9 seconds (9.5s)
2026-09-22 19:01:38 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:31<00:00, 329.50it/s]
2026-09-22 19:02:10 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:31<00:00, 329.41it/s]
2026-09-22 19:02:43 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:31<00:00, 330.43it/s]
2026-09-22 19:03:16 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:31<00:00, 329.77it/s]
2026-09

In [12]:
from stt_benchmarking.utils import deployment_protocol as dp
from stt_benchmarking.models.speech_to_text.arabic import ensemble

importlib.reload(dp)

dp = dp.DeploymentProtocol.from_hypotheses(
    model_samples={n: s for n, s in zip(model_names, all_models) if n in voters},
    voters=voters,
    ensemble_factory=ensemble.HybridEnsemble,
    main_kwargs={"substitute": False, "normalize_final_letters": True},
    n_boot=1000, seed=42, min_pool_size=3,
    outdir="deployment_out/cv_validated_voters_ensemble", suffix="cv_validated_voters_ensemble")

r = dp.run(weighting="filtration")

2026-09-22 18:37:16 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function cross_wer: 0 minutes 12 seconds (12.6s)
2026-09-22 18:37:17 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:21<00:00, 489.75it/s]
2026-09-22 18:37:38 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:21<00:00, 496.80it/s]
2026-09-22 18:38:00 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:21<00:00, 494.15it/s]
2026-09-22 18:38:22 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:21<00:00, 496.14it/s]
2026-

In [10]:
from stt_benchmarking.utils import deployment_protocol as dp
from stt_benchmarking.models.speech_to_text.arabic import ensemble
importlib.reload(dp)

dp = dp.DeploymentProtocol.from_hypotheses(
    model_samples={n: s for n, s in zip(model_names, all_models) if n in voters},
    voters=voters,
    ensemble_factory=ensemble.HybridEnsemble,
    main_kwargs={"substitute": False, "normalize_final_letters": True},
    n_boot=1000, seed=42, min_pool_size=3,
    outdir="deployment_out/cv_validated_voters_unweighted_ensemble", suffix="cv_validated_voters_unweighted_ensemble")

r = dp.run(weighting="uniform")

2026-09-22 21:33:20 - [stt_benchmarking.utils.decorators.execution_time_wrapper:21] INFO - Execution time for function cross_wer: 0 minutes 5 seconds (5.8s)
2026-09-22 21:33:20 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:21<00:00, 478.88it/s]
2026-09-22 21:33:42 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:22<00:00, 470.67it/s]
2026-09-22 21:34:05 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:37<00:00, 277.71it/s]
2026-09-22 21:34:44 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:34<00:00, 306.45it/s]
2026-09

## Rover

In [9]:
importlib.reload(rover)

ROVEROBJECT = rover.ROVER()
ROVEROBJECT.combine_models_transcriptions(seamless_samples,
    whisper_v3_samples,
    nvidia_samples,
    w2vbert_samples,
    hubert_samples,
    wav2vec_sb_samples,
    cohere_samples)
ROVEROBJECT.main(substitute=False, normalize_final_letters=True)
ROVEROBJECT.eval(cv_audios)
rover_samples = ROVEROBJECT.fusion_results
ROVEROBJECT.overall_metrics

2026-09-07 10:55:19 - [stt_benchmarking.models.speech_to_text.arabic.main:406] INFO - Using substitute: False, Replace Final Char: True
ROVER Fusion...: 100%|██████████| 10500/10500 [00:21<00:00, 489.03it/s] 


{'word_error_rate': {'wer (%)': 5.04,
  'word_similarity_ratio (%)': 97.48,
  'word_edit_distance': 2687,
  'substitutions': 2169,
  'deletions': 234,
  'insertions': 284,
  'hits': 50861,
  'ref_length': 53264,
  'hyp_length': 53314},
 'character_error_rate': {'cer (%)': 1.63,
  'char_similarity_ratio (%)': 99.19,
  'char_edit_distance': 4286,
  'substitutions': 1064,
  'deletions': 1585,
  'insertions': 1637,
  'hits': 260370,
  'ref_length': 263019,
  'hyp_length': 263071}}

In [11]:
ROVEROBJECT.fusion_wall_time

32.46507287025452

## Ensemble

In [9]:
importlib.reload(ensemble)

ENSEMBLE_OBJ = ensemble.HybridEnsemble()
ENSEMBLE_OBJ.combine_models_transcriptions(
    seamless_samples,
    whisper_v3_samples,
    nvidia_samples,
    w2vbert_samples,
    hubert_samples,
    wav2vec_sb_samples,
    cohere_samples
)
ENSEMBLE_OBJ.main(substitute=False, normalize_final_letters=True)
ENSEMBLE_OBJ.eval(cv_audios)
ensemble_samples = ENSEMBLE_OBJ.samples_info
ENSEMBLE_OBJ.overall_metrics

2026-09-07 10:57:27 - [stt_benchmarking.models.speech_to_text.arabic.main:912] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:54<00:00, 191.36it/s]


{'word_error_rate': {'wer (%)': 4.88,
  'word_similarity_ratio (%)': 97.56,
  'word_edit_distance': 2597,
  'substitutions': 2121,
  'deletions': 322,
  'insertions': 154,
  'hits': 50821,
  'ref_length': 53264,
  'hyp_length': 53096},
 'character_error_rate': {'cer (%)': 1.44,
  'char_similarity_ratio (%)': 99.28,
  'char_edit_distance': 3780,
  'substitutions': 1021,
  'deletions': 1841,
  'insertions': 918,
  'hits': 260157,
  'ref_length': 263019,
  'hyp_length': 262096}}

In [10]:
importlib.reload(ensemble)

ENSEMBLE_OBJ = ensemble.HybridEnsemble()
ENSEMBLE_OBJ.combine_models_transcriptions(
    seamless_samples,
    whisper_v3_samples,
    nvidia_samples,
    w2vbert_samples,
    hubert_samples,
    wav2vec_sb_samples,
    cohere_samples
)
ENSEMBLE_OBJ.main(substitute=False, normalize_final_letters=True)
ENSEMBLE_OBJ.eval(cv_audios)
ensemble_samples = ENSEMBLE_OBJ.samples_info
ENSEMBLE_OBJ.overall_metrics

2026-09-22 18:06:03 - [stt_benchmarking.models.speech_to_text.arabic.main:864] INFO - Using substitute: False, Replace Final Char: True
Fusing Inputs...: 100%|██████████| 10500/10500 [00:28<00:00, 372.86it/s]


{'word_error_rate': {'wer (%)': 4.88,
  'word_similarity_ratio (%)': 97.55,
  'word_edit_distance': 2601,
  'substitutions': 2121,
  'deletions': 322,
  'insertions': 158,
  'hits': 50821,
  'ref_length': 53264,
  'hyp_length': 53100},
 'character_error_rate': {'cer (%)': 1.44,
  'char_similarity_ratio (%)': 99.28,
  'char_edit_distance': 3778,
  'substitutions': 1022,
  'deletions': 1834,
  'insertions': 922,
  'hits': 260163,
  'ref_length': 263019,
  'hyp_length': 262107}}

In [10]:
ENSEMBLE_OBJ.fusion_wall_time

55.17088532447815

In [10]:
import json

def wer_of(samples, path):
    m = samples.get(path, {}).get("metrics")
    if not m: return None
    return m["word_error_rate"]["wer (%)"]

model_dicts = {
    "seamless_m4t": seamless_samples, "whisper_v3": whisper_v3_samples,
    "fastconformer": nvidia_samples, "w2vbert": w2vbert_samples,
    "hubert": hubert_samples, "wav2vec_sb": wav2vec_sb_samples,
    "cohere": cohere_samples,
}

rows = []
for path in ensemble_samples:
    ens_wer = wer_of(ensemble_samples, path)
    if ens_wer is None: continue
    sys_wers = {n: wer_of(d, path) for n, d in model_dicts.items()}
    sys_wers = {n: w for n, w in sys_wers.items() if w is not None}
    if not sys_wers: continue
    best_single = min(sys_wers.values())
    margin = best_single - ens_wer            # positive = ensemble beats all
    ref = ensemble_samples[path].get("normalized_transcription", "")
    n_words = len(ref.split())
    md = ensemble_samples[path]["metadata"]
    n_filtered = md["records_filtration"].get("num_filtered", 0)
    n_ties = sum(
        1 for vd in md["voting"]["voting_details"]
        if vd.get("token_votes") and
        sorted(vd["token_votes"].values())[-1] == (sorted(vd["token_votes"].values())[-2] if len(vd["token_votes"])>1 else -1)
    )
    rows.append({
        "path": path, "ens_wer": round(ens_wer,2), "best_single": round(best_single,2),
        "margin": round(margin,2), "n_words": n_words,
        "n_filtered": n_filtered, "n_ties": n_ties,
        "strategy": md["reference_selection"].get("strategy_metric"),
    })

# ensemble strictly beats every single system, short enough to typeset, exercises filter+tie
cand = [r for r in rows if r["margin"] > 0 and 5 <= r["n_words"] <= 12]
cand.sort(key=lambda r: (r["n_filtered"]>0, r["n_ties"]>0, r["margin"]), reverse=True)

for r in cand[:15]:
    print(r)

{'path': '../../data/mozilla_commonvoice/cv-corpus/ar/clips\\common_voice_ar_19486916.mp3', 'ens_wer': 0.0, 'best_single': 20.0, 'margin': 20.0, 'n_words': 5, 'n_filtered': 1, 'n_ties': 1, 'strategy': 'most_common_words'}
{'path': '../../data/mozilla_commonvoice/cv-corpus/ar/clips\\common_voice_ar_24030002.mp3', 'ens_wer': 0.0, 'best_single': 20.0, 'margin': 20.0, 'n_words': 5, 'n_filtered': 1, 'n_ties': 1, 'strategy': 'most_common_words'}
{'path': '../../data/mozilla_commonvoice/cv-corpus/ar/clips\\common_voice_ar_25193257.mp3', 'ens_wer': 0.0, 'best_single': 20.0, 'margin': 20.0, 'n_words': 5, 'n_filtered': 1, 'n_ties': 1, 'strategy': 'longest_match_validation'}
{'path': '../../data/mozilla_commonvoice/cv-corpus/ar/clips\\common_voice_ar_25202531.mp3', 'ens_wer': 20.0, 'best_single': 40.0, 'margin': 20.0, 'n_words': 5, 'n_filtered': 2, 'n_ties': 1, 'strategy': 'longest_match_validation'}
{'path': '../../data/mozilla_commonvoice/cv-corpus/ar/clips\\common_voice_ar_32382534.mp3', 'ens_

In [14]:
model_dicts={"seamless_m4t":seamless_samples,"whisper_v3":whisper_v3_samples,"fastconformer":nvidia_samples,"w2vbert":w2vbert_samples,"hubert":hubert_samples,"wav2vec_sb":wav2vec_sb_samples,"cohere":cohere_samples}
disp={"seamless_m4t":"SeamlessM4T v2","whisper_v3":"Whisper large-v3","fastconformer":"FastConformer-Hybrid","w2vbert":"w2v-BERT 2.0","hubert":"HuBERT","wav2vec_sb":"SpeechBrain wav2vec2","cohere":"Cohere Transcribe Arabic"}
AUDIO_ID="../../data/mozilla_commonvoice/cv-corpus/ar/clips\\common_voice_ar_24362455.mp3"

In [15]:
from stt_benchmarking.utils import ensemble_analysis
importlib.reload(ensemble_analysis)

fp=ensemble_analysis.download_amiri("fonts")     # OFL, one-time; returns Amiri-Regular.ttf
ensemble_analysis.render_stages(AUDIO_ID,ensemble_samples,model_dicts,voters,disp,ensemble_name="Proposed ensemble",fig_dir="figures_commonvoice",font_path=fp,max_tokens=14,col_chunk=12)

font: fonts\Amiri-Regular.ttf


TSIV NOT subset; don't know how to subset; dropped


saved: figures_commonvoice\stage0_reference.pdf | figures_commonvoice\stage0_reference.png


TSIV NOT subset; don't know how to subset; dropped
TSIV NOT subset; don't know how to subset; dropped


saved: figures_commonvoice\stage1_filtration.pdf | figures_commonvoice\stage1_filtration.png
saved: figures_commonvoice\stage2_reference_selection.pdf | figures_commonvoice\stage2_reference_selection.png


...

TSIV NOT subset; don't know how to subset; dropped


saved: figures_commonvoice\stage4_voting.pdf | figures_commonvoice\stage4_voting.png


TSIV NOT subset; don't know how to subset; dropped


saved: figures_commonvoice\stage5_comparison.pdf | figures_commonvoice\stage5_comparison.png


['figures_commonvoice\\stage0_reference.pdf',
 'figures_commonvoice\\stage0_reference.png',
 'figures_commonvoice\\stage1_filtration.pdf',
 'figures_commonvoice\\stage1_filtration.png',
 'figures_commonvoice\\stage2_reference_selection.pdf',
 'figures_commonvoice\\stage2_reference_selection.png',
 'figures_commonvoice\\stage3_alignment.pdf',
 'figures_commonvoice\\stage3_alignment.png',
 'figures_commonvoice\\stage4_voting.pdf',
 'figures_commonvoice\\stage4_voting.png',
 'figures_commonvoice\\stage5_comparison.pdf',
 'figures_commonvoice\\stage5_comparison.png']

In [11]:
all_models.append(rover_samples)
model_names.append("rover")

all_models.append(ensemble_samples)
model_names.append("ensemble")

In [12]:
models_with_names = list(zip(model_names, all_models))
df_full = collect_model_metrics(models_with_names)
df_filtered_full = filter_audio_records(df_full)

Audios dropped (empty ground truth): 0
Audios dropped (empty ensemble): 5
Audios: 10500 -> 10495


In [13]:
df_full_eval = evaluate_absolute_metrics(df_filtered_full)
df_full_eval[["model", "WER (%)", "CER (%)", "rtf_mean", "inference_time_total_h", "fusion_time_mean"]]

,model,WER (%),CER (%),rtf_mean,inference_time_total_h,fusion_time_mean
0,ensemble,4.81,1.37,NaN,0.125067,0.042901
1,rover,4.98,1.56,NaN,0.052627,0.018052
2,cohere,5.56,1.50,0.045209,0.556292,NaN
3,fastconformer,7.19,2.16,0.016741,0.196160,NaN
4,hubert,7.81,2.13,0.009844,0.117725,NaN
5,seamless_m4t,10.77,3.49,0.089647,1.085669,NaN
6,whisper_v3,14.71,4.64,0.160151,1.921033,NaN
7,whisper_v2,18.42,6.01,0.165181,2.003316,NaN
8,xlsr,22.82,6.43,0.007376,0.087706,NaN
9,w2vbert,23.98,6.58,0.008883,0.106979,NaN


In [ ]:
df_full_eval = evaluate_absolute_metrics(df_filtered_full)
df_full_eval[["model", "WER (%)", "CER (%)", "rtf_mean", "inference_time_total_h", "fusion_time_mean"]]

,model,WER (%),CER (%)
0,ensemble,4.81,1.37
1,rover,4.98,1.56
2,cohere,5.56,1.50
3,fastconformer,7.19,2.16
4,hubert,7.81,2.13
5,seamless_m4t,10.77,3.49
6,whisper_v3,14.71,4.64
7,whisper_v2,18.42,6.01
8,xlsr,22.82,6.43
9,w2vbert,23.98,6.58


In [14]:
df_relative_ensemble = evaluate_relative_metrics(df_filtered_full, "ensemble", exclude_models=["rover", "hybrid_ensemble"])
df_relative_ensemble[["model", "WER (%)", "CER (%)"]]

,model,WER (%),CER (%)
0,ensemble,0.00,0.00
1,cohere,3.68,1.09
2,fastconformer,5.29,1.65
3,hubert,7.52,2.05
4,seamless_m4t,8.37,2.95
5,whisper_v3,11.97,3.92
6,whisper_v2,16.41,5.49
7,xlsr,21.32,5.96
8,w2vbert,21.90,6.05
9,wav2vec_sb,27.44,9.14


# END

In [17]:
df_relative_rover = evaluate_relative_metrics(df_filtered_full, "rover", exclude_models=["ensemble", "hybrid_ensemble"])
df_relative_rover.to_csv("../../reports/phase_1_4/metrics/relative/ultimate_last/rover_relative_metrics.csv", index=False)
df_relative_ensemble = evaluate_relative_metrics(df_filtered_full, "ensemble", exclude_models=["rover", "hybrid_ensemble"])
df_relative_ensemble.to_csv("../../reports/phase_1_4/metrics/relative/ultimate_last/ensemble_relative_metrics.csv", index=False)

In [ ]:
# ---- Option B: headline ranking correlation (reference order vs pseudo-reference order) ----
merged_B, coeffs_B = ranking_correlation(df_full_eval, df_relative_ensemble)
print("=== Option B: pooled system-ranking correlation (ensemble as pseudo-reference) ===")
print(merged_B.to_string(index=False))
print(coeffs_B)

# ---- Option A: per-sample pseudo-WER tracking (NoRefER analogue) ----
df_ps = add_pseudo_reference_wer(df_filtered_full, ensemble_samples)
corr_A = per_sample_correlation(df_ps)
corr_A.to_csv("../../reports/phase_1_4/metrics/relative/ultimate_last/per_sample_correlation_cohere.csv", index=False)
print("\n=== Option A: per-sample WER tracking (per-model + pooled) ===")
print(corr_A.to_string(index=False))

=== Option B: pooled system-ranking correlation (ensemble as pseudo-reference) ===
        model  wer_abs  wer_rel
       cohere     5.56     3.83
fastconformer     7.19     5.50
       hubert     7.81     7.71
 seamless_m4t    10.77     8.47
   whisper_v3    14.71    11.88
         xlsr    22.82    20.61
      w2vbert    23.98    21.78
   wav2vec_sb    29.12    27.22
{'pearson': 0.9963394902954462, 'pearson_p': 1.222845606765019e-07, 'spearman': 1.0, 'spearman_p': 0.0, 'kendall': 0.9999999999999998, 'kendall_p': 4.96031746031746e-05}

=== Option A: per-sample WER tracking (per-model + pooled) ===
        model     n  pearson  pearson_p  spearman  spearman_p  kendall  kendall_p
   wav2vec_sb 10496 0.945278        0.0  0.948045         0.0 0.907969        0.0
      w2vbert 10496 0.936345        0.0  0.919205         0.0 0.874470        0.0
         xlsr 10496 0.922583        0.0  0.914378         0.0 0.871513        0.0
   whisper_v3 10496 0.907703        0.0  0.860109         0.0 0.819